# Notebook 2 — Train and evaluate detectors

Run Notebook 1 first; it creates `features_train.csv` and `features_test.csv`.

`report()` returns the detector metrics used throughout: accuracy, recall (share of ransomware examples caught), false positive rate (share of benign examples flagged) and precision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, roc_auc_score,
                             confusion_matrix)
from xgboost import XGBClassifier

feat_train = pd.read_csv("features_train.csv")
feat_test = pd.read_csv("features_test.csv")

META = ["label", "family", "run"]
ALL_FEATURES = [c for c in feat_train.columns if c not in META]

def make_rf(n_estimators=200):
    return RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced",
                                  n_jobs=-1, random_state=42)

def make_xgb():
    return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                         eval_metric="logloss", n_jobs=-1, random_state=42)

def report(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, y_pred),
            "recall": tp / (tp + fn) if tp + fn else np.nan,
            "false positive rate": fp / (fp + tn) if fp + tn else np.nan,
            "precision": precision_score(y_true, y_pred, zero_division=0)}

print("features used:", len(ALL_FEATURES))
print(ALL_FEATURES)

## Experiment A — Random split of the training examples

This is how the original version of this assignment evaluated its model.

In [ ]:
X, y = feat_train[ALL_FEATURES], feat_train["label"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = make_rf().fit(X_tr, y_tr)
exp_a = report(y_te, rf.predict(X_te))
pd.Series(exp_a).round(3)

## Task 2.1 — Baseline

Build the simplest detector you can that uses only **how much work the machine is doing**, without training a model. Choose its parameter using the training recordings only and evaluate it on the test recordings with `report()`. If your features do not contain what you need, load the raw recordings from `data/`.

In [ ]:
# Task 2.1

*Your explanation:*

## Experiment B — Train on the `train/` recordings, test on the `test/` recordings

In [ ]:
X_train, y_train = feat_train[ALL_FEATURES], feat_train["label"]
X_test, y_test = feat_test[ALL_FEATURES], feat_test["label"]

rf_full = make_rf().fit(X_train, y_train)
xgb_full = make_xgb().fit(X_train, y_train)

exp_b = pd.DataFrame({
    "Random Forest": {**report(y_test, rf_full.predict(X_test)),
                      "ROC-AUC": roc_auc_score(y_test, rf_full.predict_proba(X_test)[:, 1])},
    "XGBoost": {**report(y_test, xgb_full.predict(X_test)),
                "ROC-AUC": roc_auc_score(y_test, xgb_full.predict_proba(X_test)[:, 1])},
}).T.round(3)
exp_b

## Task 2.2 — Ransomware the detector has never seen

Estimate how well each model (Random Forest and XGBoost) detects a ransomware family it has never seen before. Report results for every family and overall, using the metrics you consider appropriate for a detector. Describe precisely which data trained and which data tested each model, and why.

In [ ]:
# Task 2.2

*Your explanation:*

## Task 2.3 — Alarm threshold

Choose the alarm threshold you would deploy for the Random Forest. The test recordings must not influence the choice. Report the threshold, how you chose it, and the performance it gives in the setting of Task 2.2.

In [ ]:
# Task 2.3

*Your explanation:*

## Task 2.4 — Busy or different?

Determine how much your detector depends on **how busy the machine is**, as opposed to **what kind of work it is doing**. State which features you consider activity-dependent and why.

In [ ]:
# Task 2.4

*Your explanation:*

## Task 2.5 — Feature importance

Impurity-based importance of the Random Forest from Experiment B:

In [ ]:
impurity = pd.Series(rf_full.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
impurity.plot.bar(figsize=(8, 4), title="Impurity importance")
plt.tight_layout()
plt.show()

Measure feature importance a second way. Choose the method, the data it is computed on and the metric, and justify each choice. Explain where and why the two rankings disagree.

In [ ]:
# Task 2.5

*Your explanation:*

## Task 2.6 — Why is a family missed?

Take the family your Random Forest handles worst in Task 2.2. Find evidence in the data for why it is missed.

In [ ]:
# Task 2.6

*Your explanation:*

## Summary

Compare Experiment A, Experiment B and your Task 2.2 result (at your threshold from Task 2.3) in one table, then state in a few sentences which number you would report as the detector's expected performance, and why.

In [ ]:
# Task summary

*Your explanation:*